<a href="https://colab.research.google.com/github/Kcanot/MIMICIV_demo/blob/main/notebooks/Essai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [1]:
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()

PROJECT_ID = "essai-1-489115"
SRC = "essai-1-489115.Labo"
client = bigquery.Client(project=PROJECT_ID)

In [ ]:
tables = client.query(f"""
    SELECT table_name, table_type
    FROM `{SRC}.INFORMATION_SCHEMA.TABLES`
    ORDER BY table_name
""").to_dataframe()
print(tables)

In [ ]:
cols = client.query(f"""
    SELECT table_name, column_name, data_type
    FROM `{SRC}.INFORMATION_SCHEMA.COLUMNS`
    ORDER BY table_name, ordinal_position
""").to_dataframe()
print(cols.to_string())

In [17]:
#Première jointure de test
sql = f"""
SELECT
  m.subject_id,
  m.hadm_id,
  m.micro_specimen_id,
  m.spec_type_desc,
  m.org_name,
  m.ab_name,
  m.interpretation,
  a.admission_type,
  a.admittime,
  a.dischtime
FROM `{SRC}.microbiologyevents` AS m
LEFT JOIN `{SRC}.admissions` AS a
  ON m.hadm_id = a.hadm_id
"""
df = client.query(sql).to_dataframe()

print(df.shape)
print(df["hadm_id"].isna().mean())           # part des lignes sans hospitalisation
print(df["admission_type"].value_counts(dropna=False))
print(df["interpretation"].value_counts(dropna=False))

(2899, 10)
0.3349430838220076
admission_type
None                           971
EW EMER.                       923
URGENT                         372
OBSERVATION ADMIT              295
DIRECT EMER.                   266
ELECTIVE                        60
SURGICAL SAME DAY ADMISSION     10
EU OBSERVATION                   2
Name: count, dtype: int64
interpretation
None    1863
S        818
R        190
I         28
Name: count, dtype: int64


In [ ]:
tables = ["patients", "admissions", "icustays", "prescriptions",
          "inputevents", "d_items", "microbiologyevents"]

for t in tables:
    n = client.query(f"SELECT COUNT(*) AS n FROM `{SRC}.{t}`").to_dataframe().iloc[0, 0]
    print(f"{t:20s} {n}")

patients             100
admissions           275
icustays             140
prescriptions        18087
inputevents          20404
d_items              4014
microbiologyevents   2899


In [10]:
# Vérification des types
cols = client.query(f"""
    SELECT table_name, column_name, data_type
    FROM `{SRC}.INFORMATION_SCHEMA.COLUMNS`
    WHERE table_name IN ('admissions', 'prescriptions', 'microbiologyevents', 'inputevents')
      AND (column_name LIKE '%time%' OR column_name LIKE '%date%')
    ORDER BY table_name, column_name
""").to_dataframe()
print(cols.to_string())

            table_name column_name  data_type
0           admissions   admittime  TIMESTAMP
1           admissions   deathtime  TIMESTAMP
2           admissions   dischtime  TIMESTAMP
3           admissions   edouttime  TIMESTAMP
4           admissions   edregtime  TIMESTAMP
5          inputevents     endtime  TIMESTAMP
6          inputevents   starttime  TIMESTAMP
7          inputevents   storetime  TIMESTAMP
8   microbiologyevents   chartdate  TIMESTAMP
9   microbiologyevents   charttime  TIMESTAMP
10  microbiologyevents   storedate  TIMESTAMP
11  microbiologyevents   storetime  TIMESTAMP
12       prescriptions   starttime     STRING
13       prescriptions    stoptime     STRING


In [12]:
# Correction des types pour prescriptions
sql = f"""
SELECT
  subject_id,
  hadm_id,
  drug,
  SAFE_CAST(starttime AS TIMESTAMP) AS starttime,
  SAFE_CAST(stoptime  AS TIMESTAMP) AS stoptime
FROM `{SRC}.prescriptions`
"""
check = client.query(f"""
SELECT
  COUNTIF(NULLIF(starttime, '') IS NOT NULL)                                   AS start_renseigne,
  COUNTIF(NULLIF(starttime, '') IS NOT NULL
          AND SAFE_CAST(starttime AS TIMESTAMP) IS NULL)                       AS start_non_converti,
  COUNTIF(NULLIF(stoptime, '') IS NOT NULL
          AND SAFE_CAST(stoptime AS TIMESTAMP) IS NULL)                        AS stop_non_converti
FROM `{SRC}.prescriptions`
""").to_dataframe()
print(check)

   start_renseigne  start_non_converti  stop_non_converti
0            18087                   0                  0


In [13]:
# Repérer les lignes avec antibiotiques
#inputevents
q = f"""
SELECT category, COUNT(*) AS n_items
FROM `{SRC}.d_items`
WHERE linksto = 'inputevents'
GROUP BY category
ORDER BY n_items DESC
"""
df_drugs_inputevents = client.query(q).to_dataframe()
df_drugs_inputevents


,category,n_items
0,Medications,131
1,Nutrition - Enteral,112
2,Fluids - Other (Not In Use),84
3,Antibiotics,55
4,Fluids/Intake,37
5,Blood Products/Colloids,27
6,Nutrition - Supplements,19
7,Nutrition - Parenteral,9


In [15]:
#inputevents :quels atb ?
q = f"""
SELECT itemid, label
FROM `{SRC}.d_items`
WHERE linksto = 'inputevents' AND category = 'Antibiotics'
ORDER BY label
"""
print(client.query(q).to_dataframe().to_string())

    itemid                                 label
0   225837                             Acyclovir
1   225838                              Ambisome
2   225840                              Amikacin
3   225842                            Ampicillin
4   225843         Ampicillin/Sulbactam (Unasyn)
5   225844                            Atovaquone
6   225845                          Azithromycin
7   225847                             Aztreonam
8   225899                     Bactrim (SMX/TMP)
9   225848                           Caspofungin
10  225850                             Cefazolin
11  225851                              Cefepime
12  229587                           Ceftaroline
13  225853                           Ceftazidime
14  225855                           Ceftriaxone
15  229059                       Chloramphenicol
16  225857                           Chloroquine
17  225859                         Ciprofloxacin
18  225860                           Clindamycin
19  225862          

In [14]:
#prescriptions

q = f"""
SELECT drug,
       COUNT(*) AS n_lignes,
       COUNT(DISTINCT hadm_id) AS n_admissions
FROM `{SRC}.prescriptions`
GROUP BY drug
ORDER BY n_lignes DESC
LIMIT 150
"""
df_drugs_prescriptions = client.query(q).to_dataframe()
df_drugs_prescriptions

,drug,n_lignes,n_admissions
0,Insulin,915,130
1,0.9% Sodium Chloride,810,149
2,Potassium Chloride,610,133
3,Sodium Chloride 0.9% Flush,585,242
4,Furosemide,510,103
...,...,...,...
145,Atenolol,22,12
146,Ferrous Sulfate,22,22
147,Pneumococcal Vac Polyvalent,22,15
148,Rivaroxaban,22,13


In [16]:
#prescriptions : quels atb ?
sql = f"""
SELECT
  drug,
  COUNT(*) AS n_lignes,
  COUNT(DISTINCT hadm_id) AS n_admissions
FROM `{SRC}.prescriptions`
WHERE REGEXP_CONTAINS(LOWER(drug),
  r'cillin|cef|ceph|penem|bactam|mycin|micin|amikacin|floxacin|cycline|linezolid|sulfameth|bactrim|trimethoprim|metronidazole|aztreonam|colistin|rifamp|nitrofurantoin')
GROUP BY drug
ORDER BY drug
"""
df_abx_candidats = client.query(sql).to_dataframe()
print(df_abx_candidats.to_string())

                                 drug  n_lignes  n_admissions
0               *NF* Ertapenem Sodium         2             2
1                         Amoxicillin         1             1
2         Amoxicillin-Clavulanic Acid         5             5
3                   Ampicillin Sodium         3             2
4                Ampicillin-Sulbactam        11             8
5                        Azithromycin        17            10
6                       Azithromycin          6             4
7                           Aztreonam         1             1
8                           CeFAZolin         8             8
9                         CefTAZidime        14             6
10                        CefTRIAXone        18            13
11                          CefazoLIN        41            26
12                           CefePIME        82            37
13               Cefpodoxime Proxetil         1             1
14                        CeftriaXONE        40            30
15      